In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import csr_matrix
 
pd.set_option("display.max_colwidth", 80)

In [3]:
movies= "movies.csv"
ratings= "ratings.csv"

movies = pd.read_csv(movies)
ratings = pd.read_csv(ratings)

print(movies.shape, ratings.shape)
movies.head()

(87585, 3) (32000204, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [5]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [6]:
movies["genres"] = movies["genres"].fillna("").replace("(no genres listed)", "")
movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)

In [7]:
valid_movie_ids = ratings["movieId"].unique()
movies = movies[movies["movieId"].isin(valid_movie_ids)].reset_index(drop=True)
 
movie_id_to_idx = {mid: idx for idx, mid in enumerate(movies["movieId"])}
idx_to_movie_id = {idx: mid for mid, idx in movie_id_to_idx.items()}
 
n_movies = len(movies)
n_users = ratings["userId"].nunique()
print(f"Movies: {n_movies}, Users: {n_users}")

Movies: 84432, Users: 200948


In [10]:
tfidf = TfidfVectorizer(token_pattern=r"[a-zA-Z\-]+")
tfidf_matrix = tfidf.fit_transform(movies["genres_clean"])  # sparse, stays small — fine as-is

print("TF-IDF matrix:", tfidf_matrix.shape)


TF-IDF matrix: (84432, 19)


In [11]:
user_ids = ratings["userId"].unique()
user_id_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}

row = ratings["userId"].map(user_id_to_idx)
col = ratings["movieId"].map(movie_id_to_idx)
data = ratings["rating"].astype(float)

user_item_matrix = csr_matrix((data, (row, col)), shape=(len(user_ids), n_movies))

n_components = min(50, n_movies - 1)
svd = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)          # (n_users, k) — small, fine
movie_factors = svd.components_.T                           


print("user_factors:", user_factors.shape, "movie_factors:", movie_factors.shape)

user_factors: (200948, 50) movie_factors: (84432, 50)


In [12]:
 
row = ratings["userId"].map(user_id_to_idx)
col = ratings["movieId"].map(movie_id_to_idx)
data = ratings["rating"].astype(float)

In [13]:
user_item_matrix = csr_matrix((data, (row, col)), shape=(len(user_ids), n_movies))

In [14]:
n_components = min(50, n_movies - 1)
svd = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)          # (n_users, k)
movie_factors = svd.components_.T                            # (n_movies, k)
 

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

   # (n_users, n_movies)

In [17]:
train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)



In [18]:
train_row = train_ratings["userId"].map(user_id_to_idx)
train_col = train_ratings["movieId"].map(movie_id_to_idx)
train_matrix = csr_matrix(
    (train_ratings["rating"].astype(float), (train_row, train_col)),
    shape=(len(user_ids), n_movies),
)



In [19]:
svd_eval = TruncatedSVD(n_components=n_components, random_state=42)
uf = svd_eval.fit_transform(train_matrix)
mf = svd_eval.components_.T



In [20]:
test_ratings = test_ratings[test_ratings["userId"].isin(user_id_to_idx) &
                             test_ratings["movieId"].isin(movie_id_to_idx)]



In [21]:

test_u_idx = test_ratings["userId"].map(user_id_to_idx).to_numpy()
test_m_idx = test_ratings["movieId"].map(movie_id_to_idx).to_numpy()
preds = np.sum(uf[test_u_idx] * mf[test_m_idx], axis=1)



In [22]:
rmse = np.sqrt(mean_squared_error(test_ratings["rating"], preds))
print(f"Collaborative filtering RMSE on held-out ratings: {rmse:.4f}")

Collaborative filtering RMSE on held-out ratings: 2.8213


In [23]:
rmse = np.sqrt(mean_squared_error(test_ratings["rating"], preds))
print(f"Collaborative filtering RMSE on held-out ratings: {rmse:.4f}")
 

Collaborative filtering RMSE on held-out ratings: 2.8213


In [24]:
scaler = MinMaxScaler()

In [25]:
def hybrid_score(user_id, alpha=0.6, liked_movie_ids=None):
    if user_id not in user_id_to_idx:
        raise ValueError(f"user_id {user_id} not found in training data")

    u_idx = user_id_to_idx[user_id]

In [31]:
def hybrid_score(user_id, alpha=0.6, liked_movie_ids=None):
    if user_id not in user_id_to_idx:
        raise ValueError(f"user_id {user_id} not found in training data")

    u_idx = user_id_to_idx[user_id]

    # ---- Collaborative component ----
    cf_scores = user_factors[u_idx].dot(movie_factors.T)
    cf_scores_norm = scaler.fit_transform(cf_scores.reshape(-1, 1)).flatten()

    # ---- Content component ----
    if liked_movie_ids is None:
        user_hist = ratings[ratings["userId"] == user_id].sort_values("rating", ascending=False)
        liked_movie_ids = user_hist["movieId"].head(10).tolist()

    liked_idxs = [movie_id_to_idx[m] for m in liked_movie_ids if m in movie_id_to_idx]

    if liked_idxs:
        content_scores = cosine_similarity(tfidf_matrix[liked_idxs], tfidf_matrix).mean(axis=0)
        content_scores_norm = scaler.fit_transform(content_scores.reshape(-1, 1)).flatten()
    else:
        content_scores_norm = np.zeros(n_movies)

    fused = alpha * cf_scores_norm + (1 - alpha) * content_scores_norm
    return fused


In [32]:
def recommend_movies(user_id, top_n=10, alpha=0.6, exclude_seen=True):
    scores = hybrid_score(user_id, alpha=alpha)
 
    result = movies.copy()
    result["hybrid_score"] = scores
 
    if exclude_seen:
        seen_ids = set(ratings.loc[ratings["userId"] == user_id, "movieId"])
        result = result[~result["movieId"].isin(seen_ids)]
 
    result = result.sort_values("hybrid_score", ascending=False).head(top_n)
    return result[["movieId", "title", "genres", "hybrid_score"]].reset_index(drop=True)

In [33]:
sample_user = ratings["userId"].iloc[0]
recommend_movies(sample_user, top_n=10, alpha=0.6)

,movieId,title,genres,hybrid_score
0,2858,American Beauty (1999),Drama|Romance,0.708446
1,1704,Good Will Hunting (1997),Drama|Romance,0.689364
2,1193,One Flew Over the Cuckoo's Nest (1975),Drama,0.678141
3,858,"Godfather, The (1972)",Crime|Drama,0.668460
4,1244,Manhattan (1979),Comedy|Drama|Romance,0.668072
5,58,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,0.667773
6,903,Vertigo (1958),Drama|Mystery|Romance|Thriller,0.662364
7,1222,Full Metal Jacket (1987),Drama|War,0.657794
8,920,Gone with the Wind (1939),Drama|Romance|War,0.647809
9,750,Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964),Comedy|War,0.646103


In [34]:
import joblib

In [35]:



joblib.dump(tfidf, "tfidf_vectorizer.joblib")
joblib.dump(svd, "svd_model.joblib")
joblib.dump(tfidf_matrix, "tfidf_matrix.joblib")       # sparse, small
np.save("user_factors.npy", user_factors)               # small
np.save("movie_factors.npy", movie_factors)             # small
movies.to_pickle("movies_processed.pkl")



In [36]:
 

print("Saved: tfidf_vectorizer.joblib, svd_model.joblib, tfidf_matrix.joblib, "
      "user_factors.npy, movie_factors.npy, movies_processed.pkl")

Saved: tfidf_vectorizer.joblib, svd_model.joblib, tfidf_matrix.joblib, user_factors.npy, movie_factors.npy, movies_processed.pkl
